In [10]:
import numpy as np
from sklearn.metrics import accuracy_score
from keras.datasets import reuters
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN,Input

# Load Reuters dataset
(X_train, y_train), (X_test, y_test) = reuters.load_data(num_words=30000)

# Convert output labels into one-hot vectors
y_train = to_categorical(y_train, 46)
y_test = to_categorical(y_test, 46)

# Make all input sequences same length
X_train = pad_sequences(X_train, maxlen=50)
X_test = pad_sequences(X_test, maxlen=50)

# Convert into 3D shape for RNN
X_train = X_train.reshape(X_train.shape[0], 50, 1)
X_test = X_test.reshape(X_test.shape[0], 50, 1)

# Build RNN model
model = Sequential()
model.add(Input(shape=(50,1)))
model.add(SimpleRNN(50))
model.add(Dense(46, activation='softmax'))

model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, batch_size=50)

loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3348 - loss: 2.5099
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3513 - loss: 2.4071
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3510 - loss: 2.3937
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3525 - loss: 2.3659
Epoch 5/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3564 - loss: 2.3525
Epoch 6/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3583 - loss: 2.3397
Epoch 7/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3605 - loss: 2.3343
Epoch 8/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3598 - loss: 2.3253
Epoch 9/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3607 - loss: 2.3200
Epoch 10/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3614 - loss: 2.3102
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 588us/step - accuracy: 0.3691 - loss: 2.3104
Test Loss: 2.3104114532470703, Test Accuracy: 0.3691006302833557

In [4]:
# RNN for Text Generation

# Import libraries
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Sample text data
text = """
deep learning is very important for machine learning
deep learning is a part of artificial intelligence
rnn is useful for sequence prediction
text generation using rnn is interesting
machine learning models learn patterns from data
artificial intelligence is transforming industries
neural networks are powerful models
deep learning helps in image processing
natural language processing uses rnn networks
recurrent neural networks process sequential data
"""

# ---------------- TOKENIZATION ----------------

# Create tokenizer object
tokenizer = Tokenizer()

# Learn all words from text
tokenizer.fit_on_texts([text])

# Total number of unique words
total_words = len(tokenizer.word_index) + 1

# Convert text into sequence of numbers
token_list = tokenizer.texts_to_sequences([text])[0]

# ---------------- CREATE INPUT SEQUENCES ----------------

input_sequences = []

# Create sequences step by step
for i in range(1, len(token_list)):
    seq = token_list[:i + 1]
    input_sequences.append(seq)

# Find maximum sequence length
max_len = max(len(seq) for seq in input_sequences)

# Add padding at beginning
input_sequences = pad_sequences(input_sequences,
                                maxlen=max_len,
                                padding='pre')

# ---------------- SPLIT X AND y ----------------

# All columns except last word
X = input_sequences[:, :-1]

# Last word is output
y = input_sequences[:, -1]

# Convert output into categorical format
y = to_categorical(y, num_classes=total_words)

# ---------------- BUILD RNN MODEL ----------------

model = Sequential()

# Convert word numbers into vectors
model.add(Embedding(total_words, 50))

# RNN layer
model.add(SimpleRNN(150))

# Output layer
model.add(Dense(total_words, activation='softmax'))

# Compile model
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Train model
model.fit(X, y, epochs=300)

# ---------------- TEXT GENERATION ----------------

# Starting words
seed_text = "deep learning"

# Generate 10 new words
for i in range(10):

    # Convert words to numbers
    token = tokenizer.texts_to_sequences([seed_text])[0]

    # Add padding
    token = pad_sequences([token],
                          maxlen=max_len - 1,
                          padding='pre')

    # Predict next word
    prediction = model.predict(token, verbose=0)

    # Get word index with highest probability
    predicted_word_index = np.argmax(prediction)

    # Find actual word from index
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            next_word = word
            break

    # Add predicted word to sentence
    seed_text = seed_text + " " + next_word

# Print generated text
print(seed_text)

Epoch 1/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.0484 - loss: 3.7316 
Epoch 2/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1129 - loss: 3.5866
Epoch 3/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1774 - loss: 3.4602
Epoch 4/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2258 - loss: 3.3444
Epoch 5/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3710 - loss: 3.3159
Epoch 6/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4032 - loss: 3.1457
Epoch 7/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4516 - loss: 3.0652
Epoch 8/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4839 - loss: 2.9600
Epoch 9/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5161 - loss: 2.8524
Epoch 10/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5484 - loss: 2.7533
Epoch 11/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5484 - loss: 2.6337
Epoch 12/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6129 - l